In [1]:
from datetime import date

import pandas as pd
import requests

series_id = 433
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{series_id}/dados"
params = {
    "formato": "json",
    "dataInicial": "01/01/2000",
    "dataFinal": date.today().strftime("%d/%m/%Y")
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
records = response.json()

records[:3]

[{'data': '01/01/2000', 'valor': '0.62'},
 {'data': '01/02/2000', 'valor': '0.13'},
 {'data': '01/03/2000', 'valor': '0.22'}]

In [4]:
df_raw = pd.DataFrame(records)
df_raw.info()
df_raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   data    322 non-null    str  
 1   valor   322 non-null    str  
dtypes: str(2)
memory usage: 5.2 KB


,data,valor
0,01/01/2000,0.62
1,01/02/2000,0.13
2,01/03/2000,0.22
3,01/04/2000,0.42
4,01/05/2000,0.01


In [9]:
reference_month = pd.to_datetime(
    df_raw["data"], format="%d/%m/%Y"
).dt.to_period("M")

observed_months = pd.PeriodIndex(reference_month.dropna().unique()).sort_values()
expected_months = pd.period_range("2000-01", observed_months.max(), freq="M")
missing_months = expected_months.difference(observed_months)

pd.Series({
    "Total de linhas": len(df_raw),
    "Primeiro mês": observed_months.min(),
    "Último mês": observed_months.max(),
    "Datas ausentes": reference_month.isna().sum(),
    "Linhas excedentes no mesmo mês": reference_month.dropna().duplicated().sum(),
    "Meses ausentes desde jan/2000": missing_months.tolist(),
    "Meses distintos em 2024": reference_month[reference_month.dt.year == 2024].nunique(),
})

Total de linhas                       322
Primeiro mês                      2000-01
Último mês                        2026-10
Datas ausentes                          0
Linhas excedentes no mesmo mês          0
Meses ausentes desde jan/2000          []
Meses distintos em 2024                12
dtype: object

In [10]:
value_numeric = pd.to_numeric(df_raw["valor"], errors="coerce")

pd.Series({
    "Valores ausentes na origem": df_raw["valor"].isna().sum(),
    "Falhas de conversão": (df_raw["valor"].notna() & value_numeric.isna()).sum(),
    "Menor variação mensal (%)": value_numeric.min(),
    "Maior variação mensal (%)": value_numeric.max(),
    "Meses com variação zero": value_numeric.eq(0).sum(),
    "Meses com variação negativa": value_numeric.lt(0).sum(),
}).to_frame("resultado")

,resultado
Valores ausentes na origem,0.00
Falhas de conversão,0.00
Menor variação mensal (%),-0.68
Maior variação mensal (%),3.02
Meses com variação zero,1.00
Meses com variação negativa,16.00
